<a href="https://colab.research.google.com/github/dymiyata/python-pro-summer-2026/blob/main/logistic_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Logistic Regression

Here we learn how to perform logistic regression using sklearn

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Now we import the dataset we will use.  This is a dataset built in to sklearn.  It has data on a bunch of tumors.  

- We will load all the features into a DataFrame called `X`.  
- The target variable `y` is whether or not the tumor is malignant (cancerous)
  - We have a value of `1` for malignant
  - We have a value of `0` for not malignant (aka benign).
- We'll load the values of the target variable into `y`

In [2]:
from sklearn.datasets import load_breast_cancer
cancer = load_breast_cancer()

X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target)


In [3]:
X.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [7]:
y.tail()

,0
564,0
565,0
566,0
567,0
568,1


### Split into Training and Testing sets

Now we must split the data into a training set and a testing set.  Let's do a 75/25 train test split.  We'll use 2026 for the `random_state`.

- We'll need to add one more argument to `train_test_split` which is `stratify=y`.  What do you think this does?

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=2026,
    stratify=y # ensures proportion of y values is same in both datasets
)

`stratify=y` just ensures the proportion of 0s and 1s is about the same in the training and testing sets.

### Preprocessing the Data

Recall, in linear regression it's important for your features to be similar in scale. Why?

The same holds true for logistic regression.  Before I had you just manually and imprecisely scale the features, there is a better way to do things.



To do this, we will take each feature variable $x$ then:
- subtract its mean $\bar{x}$ from each training example so that the mean gets shifted to $0$
- divide each training example by the standard deviation $s$ so that the standard deviation becomes 1.

In this way, you will replace each feature $x$ with $$\frac{x - \bar{x}}{s}.$$  Thus, each resulting feature will have mean 0 and standard deviation 1 so they will all be on the same scale.

Luckily, `sklearn` has a built-in way to do this automatically:

In [9]:
from sklearn.preprocessing import StandardScaler

In [11]:
scaler = StandardScaler()

In [12]:
scaler.fit(X_train)

StandardScaler()

In [14]:
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [15]:
X_train.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
288,11.260,19.96,73.72,394.1,0.08020,0.11810,0.092740,0.05588,0.2595,0.06233,...,11.86,22.33,78.27,437.6,0.1028,0.1843,0.15460,0.09314,0.2955,0.07009
424,9.742,19.12,61.93,289.7,0.10750,0.08333,0.008934,0.01967,0.2538,0.07029,...,11.21,23.17,71.79,380.9,0.1398,0.1352,0.02085,0.04589,0.3196,0.08009
208,13.110,22.54,87.02,529.4,0.10020,0.14830,0.087050,0.05102,0.1850,0.07310,...,14.55,29.16,99.48,639.3,0.1349,0.4402,0.31620,0.11260,0.4128,0.10760
171,13.430,19.63,85.84,565.4,0.09048,0.06288,0.058580,0.03438,0.1598,0.05671,...,17.98,29.87,116.60,993.6,0.1401,0.1546,0.26440,0.11600,0.2884,0.07371
235,14.030,21.25,89.79,603.4,0.09070,0.06945,0.014620,0.01896,0.1517,0.05835,...,15.33,30.28,98.27,715.5,0.1287,0.1513,0.06231,0.07963,0.2226,0.07617


In [17]:
X_train_scaled

array([[-0.79043824,  0.13774499, -0.72878826, ..., -0.31250645,
         0.13296378, -0.76361944],
       [-1.21147418, -0.05213392, -1.20310801, ..., -1.02623386,
         0.55098372, -0.20595589],
       [-0.27731801,  0.72094453, -0.19372018, ..., -0.01855649,
         2.16755875,  1.32817656],
       ...,
       [-0.37439481, -1.35190033, -0.38401131, ..., -0.44890769,
        -0.54523455, -0.06988598],
       [-1.07057414, -0.71671014, -1.01844917, ..., -0.24755347,
        -0.46891555,  0.73816852],
       [ 1.95544838,  0.83396769,  1.82505547, ...,  1.31524523,
        -0.09772772, -0.47530738]])

### Training the Logistic Regression model

Now that we've split the data into training and testing sets and preprocessed the data, we're ready to train the model.  

This part will be almost the same as it was for linear regression:

First we import and define the logistic regression model:

In [18]:
from sklearn.linear_model import LogisticRegression

Now we define the model and fit it to the data.

In [19]:
model = LogisticRegression()

In [21]:
model.fit(X_train_scaled, y_train)

LogisticRegression()

In [22]:
# access weights and bias:
print(model.coef_)
print(model.intercept_)

[[-0.38488969 -0.24142965 -0.35756775 -0.387094   -0.10489287  0.37624153
  -0.72028488 -1.01869013  0.26677586  0.36084732 -1.19475676 -0.1271353
  -0.69695894 -0.76290096  0.09494148  0.85487509  0.26351407 -0.72664418
   0.1195997   0.7340352  -0.89132002 -1.14636154 -0.7234767  -0.79045025
  -0.67474185 -0.18527954 -0.92032031 -0.88271546 -0.68597375 -0.60076625]]
[0.25403613]


With logistic regression, our model predicts probabilities that $y=1$ meaning probabilities that tumors are malignant.  

We can access these values as follows:

In [26]:
y_train_prob = model.predict_proba(X_train_scaled) # gives both prob for 0 and for 1 in two separate columns
y_test_prob = model.predict_proba(X_test_scaled)

In [29]:
# print(y_train_prob);

To get the actual predictions (just 0 or 1), we compare the probabilities to a threshold value (by defualt we use a threshould of 0.5).
- values above the threshold are predicted to be 1.
-values below the threshould are predicted to be 0.

`sklearn` does this automatically with just `model.predict()` where it uses a threshould of 0.5

In [27]:
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

In [28]:
print(y_train_pred)

[1 1 1 0 1 0 0 1 1 1 1 0 0 1 1 1 1 1 0 1 1 1 1 1 1 0 1 1 0 1 1 0 1 1 1 1 0
 1 1 0 0 1 1 0 1 1 1 1 1 1 1 0 1 0 0 0 1 1 0 1 1 0 0 1 1 1 1 0 1 0 0 1 1 1
 1 0 0 1 1 0 1 0 1 1 1 1 1 1 0 0 0 1 1 1 1 1 1 0 1 0 0 1 1 1 1 1 1 1 0 1 0
 1 0 1 1 0 0 1 0 0 1 0 1 1 1 0 1 1 0 1 0 1 1 1 1 1 1 1 1 1 0 1 0 0 1 1 1 1
 1 0 0 0 1 1 1 1 0 1 0 0 1 1 1 0 0 0 1 1 0 0 1 1 1 1 0 1 1 1 1 1 0 1 0 1 0
 1 1 0 1 0 0 1 1 0 0 1 0 1 1 0 1 1 0 0 1 1 0 0 1 0 0 0 1 0 1 1 0 1 1 0 1 0
 1 0 1 0 1 1 0 1 0 1 1 0 0 1 1 1 1 0 1 0 1 0 1 1 1 0 0 0 0 0 1 1 1 1 1 0 1
 0 1 0 0 1 1 1 0 1 1 0 1 0 1 1 1 1 1 0 0 1 0 0 0 0 1 1 0 1 1 1 0 1 1 1 1 0
 1 1 0 0 1 1 1 1 0 1 1 1 1 1 0 1 1 1 0 1 1 0 1 0 1 1 1 1 1 0 0 1 0 0 0 1 1
 0 1 1 1 1 0 0 1 1 1 0 1 0 1 1 0 1 1 0 0 1 1 1 0 0 1 0 1 0 1 1 0 1 1 0 1 1
 1 1 1 0 0 1 1 0 1 1 1 1 1 1 1 1 0 1 1 1 1 0 1 1 0 0 1 0 0 1 1 0 1 1 0 0 0
 0 1 1 1 1 1 0 1 0 0 1 1 0 1 1 1 1 1 0]


### Evaluating the model

Now how do we evaluate the model?

A good way is to check our accuracy on the test set.
- Accuracy just means what percentage of examples we classify correctly.

`sklearn` has a nice function to help us compute this called `accuracy_score`.  Let's import it and use it:

In [30]:
from sklearn.metrics import accuracy_score

In [33]:
print(accuracy_score(y_train, y_train_pred))
print(accuracy_score(y_test, y_test_pred))

0.9906103286384976
0.972027972027972


Another nice visualization is something called the "Confusion Matrix".  Let's see how that works

In [35]:
from sklearn.metrics import confusion_matrix

In [37]:
print(confusion_matrix(y_train, y_train_pred))

[[155   4]
 [  0 267]]


The confusion matrix outputs a $2\times 2$ matrix showing True/False Positives/Negatives.  The order goes:
$$\begin{bmatrix}
\text{TN} & \text{FP} \\
\text{FN} & \text{TP}
\end{bmatrix}$$

I always forget the order and have to look it up...